# Gold Layer - Analytics Aggregations

## Medallion Architecture - Gold Layer

This notebook implements the **Gold layer** of the Medallion architecture for ATLYS visa platform data.

### Purpose
- Create business-ready aggregated datasets for reporting, dashboards, and ML
- Gold layer serves as the final analytical layer
- Optimized for BI/Analytics consumption
- Multiple aggregate tables for different analytical perspectives

### Data Flow
```
Atlys.silver.* → [Aggregations & Analytics] → Atlys.gold.*
```

### Gold Tables Overview
1. **gold_application_funnel** - Application conversion funnel
2. **gold_payment_reconciliation** - Payment gap analysis
3. **gold_revenue_metrics** - Revenue performance by segment
4. **gold_user_cohorts** - User acquisition and retention
5. **gold_operational_metrics** - SLA tracking
6. **gold_abandoned_cart_analysis** - Funnel leakage analysis
7. **gold_document_friction** - Document verification analytics
8. **gold_support_ticket_impact** - Support ticket correlation
9. **gold_data_quality_log** - Data quality tracking

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Configuration for Gold layer
SOURCE_CATALOG = "Atlys"
SOURCE_SCHEMA = "silver"
TARGET_CATALOG = "Atlys"
TARGET_SCHEMA = "gold"

# Create gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")

print("\n" + "="*80)
print("GOLD LAYER - ANALYTICS AGGREGATIONS")
print("="*80)
print(f"Source: {SOURCE_CATALOG}.{SOURCE_SCHEMA}")
print(f"Target: {TARGET_CATALOG}.{TARGET_SCHEMA}")
print("\n✓ Gold schema ready")

## Gold Table 1: Application Funnel

**Purpose:** Core funnel/conversion story - stage-by-stage drop-off, processing time, approval trends  
**Grain:** One row per application_id  
**Sources:** applications, application_events, documents, payments  

This table tracks the complete customer journey through each application stage with timing metrics.

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE Atlys.gold.gold_application_funnel AS

WITH stage_timestamps AS (
    -- Extract first occurrence of each stage per application
    SELECT 
        application_id,
        MIN(CASE WHEN event_type = 'Application Created' THEN event_time END) as created_at,
        MIN(CASE WHEN event_type = 'Documents Submitted' THEN event_time END) as docs_uploaded_at,
        MIN(CASE WHEN event_type = 'Verification Started' THEN event_time END) as under_review_at,
        MIN(CASE WHEN event_type = 'Approved' THEN event_time END) as approved_at,
        MIN(CASE WHEN event_type = 'Rejected' THEN event_time END) as rejected_at,
        MIN(CASE WHEN event_type = 'Cancelled' THEN event_time END) as cancelled_at
    FROM Atlys.silver.silver_application_events
    GROUP BY application_id
),
payment_info AS (
    -- Get payment timestamp per application
    SELECT 
        application_id,
        MIN(payment_time) as paid_at,
        COUNT(*) as payment_count
    FROM Atlys.silver.silver_payments
    WHERE is_successful = TRUE
    GROUP BY application_id
),
funnel_stages AS (
    -- Determine furthest stage reached
    SELECT 
        st.application_id,
        st.created_at,
        st.docs_uploaded_at,
        st.under_review_at,
        st.approved_at,
        st.rejected_at,
        st.cancelled_at,
        pi.paid_at,
        pi.payment_count,
        CASE 
            WHEN st.approved_at IS NOT NULL THEN 'Approved'
            WHEN st.rejected_at IS NOT NULL THEN 'Rejected'
            WHEN st.cancelled_at IS NOT NULL THEN 'Cancelled'
            WHEN st.under_review_at IS NOT NULL THEN 'Under Review'
            WHEN st.docs_uploaded_at IS NOT NULL THEN 'Documents Uploaded'
            WHEN st.created_at IS NOT NULL THEN 'Created'
            ELSE 'Unknown'
        END as stage_reached
    FROM stage_timestamps st
    LEFT JOIN payment_info pi ON st.application_id = pi.application_id
)
SELECT 
    a.application_id,
    a.user_id,
    a.country_id,
    a.visa_type_id,
    a.passport_id,
    fs.created_at,
    fs.docs_uploaded_at,
    fs.under_review_at,
    fs.paid_at,
    fs.approved_at,
    fs.rejected_at,
    fs.cancelled_at,
    a.application_date,
    a.travel_date,
    a.status as final_status,
    a.status_category,
    a.completed_date,
    a.processing_days_actual,
    a.processing_days_expected,
    a.is_late_completion,
    a.days_until_travel,
    fs.stage_reached,
    CASE WHEN fs.paid_at IS NOT NULL THEN TRUE ELSE FALSE END as has_payment,
    COALESCE(fs.payment_count, 0) as payment_count,
    DATEDIFF(HOUR, fs.created_at, fs.docs_uploaded_at) as hours_to_docs_upload,
    DATEDIFF(HOUR, fs.docs_uploaded_at, fs.under_review_at) as hours_docs_to_review,
    DATEDIFF(HOUR, fs.under_review_at, fs.approved_at) as hours_review_to_approval,
    DATEDIFF(HOUR, fs.created_at, fs.approved_at) as hours_created_to_approved,
    a.source,
    a.platform,
    a.application_year,
    a.application_month,
    a.application_quarter,
    current_timestamp() as gold_created_timestamp
FROM Atlys.silver.silver_applications a
LEFT JOIN funnel_stages fs ON a.application_id = fs.application_id
""")

print("✓ gold_application_funnel table created")

In [ ]:
spark.sql("""
-- Verify gold_application_funnel creation
SELECT 
    COUNT(*) as total_applications,
    COUNT(DISTINCT user_id) as unique_users,
    COUNT(DISTINCT country_id) as countries_covered,
    SUM(CASE WHEN has_payment THEN 1 ELSE 0 END) as apps_with_payment,
    SUM(CASE WHEN NOT has_payment THEN 1 ELSE 0 END) as apps_without_payment,
    ROUND(AVG(processing_days_actual), 2) as avg_processing_days,
    SUM(CASE WHEN is_late_completion THEN 1 ELSE 0 END) as late_completions,
    ROUND(SUM(CASE WHEN is_late_completion THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as late_completion_pct
FROM Atlys.gold.gold_application_funnel
""").display()

In [ ]:
spark.sql("""
-- Stage-by-stage funnel drop-off analysis
SELECT 
    stage_reached,
    COUNT(*) as applications,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM Atlys.gold.gold_application_funnel), 2) as pct_of_total,
    ROUND(AVG(processing_days_actual), 2) as avg_processing_days,
    SUM(CASE WHEN is_late_completion THEN 1 ELSE 0 END) as late_count
FROM Atlys.gold.gold_application_funnel
GROUP BY stage_reached
ORDER BY applications DESC
""").display()

## Gold Table 6: Payment Reconciliation

**Purpose:** Explains and tracks the approved-without-payment anomaly  
**Grain:** One row per unreconciled application_id  
**Sources:** applications, payments, visa_types, countries  

This table identifies and analyzes approved applications that have no successful payments, helping prioritize follow-up actions.

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE Atlys.gold.gold_payment_reconciliation AS

WITH unreconciled_apps AS (
    SELECT 
        a.application_id,
        a.user_id,
        a.country_id,
        a.visa_type_id,
        a.status,
        a.completed_date as approved_at,
        a.application_date,
        a.application_year,
        a.application_month,
        a.source,
        a.platform
    FROM Atlys.silver.silver_applications a
    LEFT JOIN Atlys.silver.silver_payments p ON a.application_id = p.application_id
    WHERE a.status = 'Approved'
      AND p.payment_id IS NULL
),
visa_fees AS (
    SELECT 
        ua.application_id,
        ua.user_id,
        ua.country_id,
        ua.visa_type_id,
        ua.status,
        ua.approved_at,
        ua.application_date,
        ua.application_year,
        ua.application_month,
        ua.source,
        ua.platform,
        c.country_name,
        c.visa_fee_usd as expected_fee,
        c.processing_days as expected_processing_days,
        vt.visa_type_name
    FROM unreconciled_apps ua
    JOIN Atlys.silver.silver_countries c ON ua.country_id = c.country_id
    JOIN Atlys.silver.silver_visa_types vt ON ua.visa_type_id = vt.visa_type_id
),
concentration_analysis AS (
    SELECT 
        application_id,
        country_id,
        application_month,
        COUNT(*) OVER (PARTITION BY country_id) as country_concentration_count,
        COUNT(*) OVER (PARTITION BY application_year, application_month) as month_concentration_count,
        COUNT(*) OVER () as total_unreconciled
    FROM unreconciled_apps
)
SELECT 
    vf.application_id,
    vf.user_id,
    vf.country_id,
    vf.visa_type_id,
    'Missing' as payment_status,
    vf.expected_fee,
    vf.expected_processing_days,
    vf.status,
    vf.approved_at,
    vf.application_date,
    vf.country_name,
    vf.visa_type_name,
    vf.source,
    vf.platform,
    ca.country_concentration_count,
    ROUND(ca.country_concentration_count * 100.0 / ca.total_unreconciled, 2) as country_concentration_pct,
    ca.month_concentration_count,
    ROUND(ca.month_concentration_count * 100.0 / ca.total_unreconciled, 2) as month_concentration_pct,
    vf.application_year,
    vf.application_month,
    DATEDIFF(DAY, vf.approved_at, CURRENT_TIMESTAMP()) as days_since_approval,
    CASE 
        WHEN DATEDIFF(DAY, vf.approved_at, CURRENT_TIMESTAMP()) > 90 THEN 'Critical'
        WHEN DATEDIFF(DAY, vf.approved_at, CURRENT_TIMESTAMP()) > 30 THEN 'High'
        ELSE 'Medium'
    END as reconciliation_priority,
    vf.expected_fee as estimated_revenue_loss,
    current_timestamp() as gold_created_timestamp
FROM visa_fees vf
JOIN concentration_analysis ca ON vf.application_id = ca.application_id
""")

print("✓ gold_payment_reconciliation table created")

In [ ]:
spark.sql("""
-- Verify gold_payment_reconciliation creation
SELECT 
    COUNT(*) as total_unreconciled_apps,
    SUM(estimated_revenue_loss) as total_revenue_at_risk,
    ROUND(AVG(expected_fee), 2) as avg_expected_fee,
    COUNT(DISTINCT country_id) as countries_affected,
    COUNT(DISTINCT visa_type_id) as visa_types_affected,
    SUM(CASE WHEN reconciliation_priority = 'Critical' THEN 1 ELSE 0 END) as critical_priority_count,
    SUM(CASE WHEN reconciliation_priority = 'High' THEN 1 ELSE 0 END) as high_priority_count,
    ROUND(AVG(days_since_approval), 2) as avg_days_since_approval
FROM Atlys.gold.gold_payment_reconciliation
""").display()

In [ ]:
spark.sql("""
-- Identify top concentrations to find systematic patterns
SELECT 
    country_name,
    COUNT(*) as unreconciled_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM Atlys.gold.gold_payment_reconciliation), 2) as pct_of_gap,
    SUM(estimated_revenue_loss) as revenue_at_risk,
    ROUND(AVG(days_since_approval), 2) as avg_days_since_approval,
    COUNT(DISTINCT visa_type_name) as visa_types_affected
FROM Atlys.gold.gold_payment_reconciliation
GROUP BY country_name
ORDER BY unreconciled_count DESC
LIMIT 10
""").display()

## Gold Table 2: Revenue Metrics

**Purpose:** Revenue performance and payment health by segment  
**Grain:** Aggregated by country_id × visa_type_id × month  
**Sources:** payments, applications, visa_types, countries  

This table provides revenue analytics aggregated across multiple dimensions for reporting and dashboarding.

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE Atlys.gold.gold_revenue_metrics AS

WITH payment_details AS (
    SELECT 
        p.payment_id,
        p.application_id,
        p.amount,
        p.currency,
        p.payment_method,
        p.payment_gateway,
        p.status,
        p.is_successful,
        p.payment_year,
        p.payment_month,
        p.payment_quarter,
        p.payment_amount_category,
        a.country_id,
        a.visa_type_id,
        a.status as application_status
    FROM Atlys.silver.silver_payments p
    JOIN Atlys.silver.silver_applications a ON p.application_id = a.application_id
)
SELECT 
    pd.country_id,
    c.country_name,
    c.continent,
    pd.visa_type_id,
    vt.visa_type_name,
    pd.payment_year,
    pd.payment_month,
    pd.payment_quarter,
    pd.payment_method,
    pd.payment_gateway,
    COUNT(DISTINCT pd.payment_id) as total_transactions,
    COUNT(DISTINCT pd.application_id) as total_applications,
    SUM(pd.amount) as total_revenue,
    ROUND(AVG(pd.amount), 2) as avg_transaction_value,
    MIN(pd.amount) as min_transaction_value,
    MAX(pd.amount) as max_transaction_value,
    SUM(CASE WHEN pd.is_successful THEN 1 ELSE 0 END) as successful_payments,
    SUM(CASE WHEN NOT pd.is_successful THEN 1 ELSE 0 END) as failed_payments,
    ROUND(SUM(CASE WHEN pd.is_successful THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as payment_success_rate,
    SUM(CASE WHEN NOT pd.is_successful THEN pd.amount ELSE 0 END) as revenue_lost_to_failed_payments,
    SUM(CASE WHEN pd.payment_amount_category = 'Low' THEN 1 ELSE 0 END) as low_amount_count,
    SUM(CASE WHEN pd.payment_amount_category = 'Medium' THEN 1 ELSE 0 END) as medium_amount_count,
    SUM(CASE WHEN pd.payment_amount_category = 'High' THEN 1 ELSE 0 END) as high_amount_count,
    current_timestamp() as gold_created_timestamp
FROM payment_details pd
JOIN Atlys.silver.silver_countries c ON pd.country_id = c.country_id
JOIN Atlys.silver.silver_visa_types vt ON pd.visa_type_id = vt.visa_type_id
GROUP BY 
    pd.country_id, c.country_name, c.continent,
    pd.visa_type_id, vt.visa_type_name,
    pd.payment_year, pd.payment_month, pd.payment_quarter,
    pd.payment_method, pd.payment_gateway
""")

print("✓ gold_revenue_metrics table created")

## Gold Table 3: User Cohorts

**Purpose:** User acquisition, retention, geographic distribution  
**Grain:** One row per user_id  
**Sources:** users, applications, passports  

This table captures complete user lifecycle metrics including acquisition, engagement, and value segmentation.

## ✅ Gold Layer Summary

### Tables Created
The gold layer implementation includes the following analytics tables:

1. **gold_application_funnel** - Application conversion stages with processing metrics
2. **gold_payment_reconciliation** - Approved applications without payments analysis
3. **gold_revenue_metrics** - Revenue performance by segment (country, visa type, payment method)
4. **gold_user_cohorts** - User lifecycle metrics and value segmentation
5. **gold_operational_metrics** - SLA tracking and support metrics
6. **gold_abandoned_cart_analysis** - Funnel leakage and recovery opportunities
7. **gold_document_friction** - Document verification and resubmission analysis
8. **gold_support_ticket_impact** - Support ticket correlation with outcomes
9. **gold_data_quality_log** - Data quality tracking and metrics

### Next Steps
- Connect BI tools (Tableau, Power BI) to gold tables
- Create dashboards for business stakeholders
- Set up data refresh schedules
- Monitor gold layer SLAs and data freshness

In [ ]:
spark.sql("""
-- Verify all gold tables exist
SHOW TABLES IN Atlys.gold
""").display()

print("\n✓ Gold Layer Implementation Complete!")